# Temporal Data: Missing Feature Investigation

The DRB temporal output has 631 of 765 HRUs — 134 features missing (17.5%).  
This notebook investigates which features are missing and why.

**Key question:** Is this a batching/reassembly bug, a gdptools WeightGen coverage issue,
or expected behavior for small polygons that don't overlap the ~4km gridMET grid?

In [ ]:
import os

# Ensure PROJ_DATA is set for the Jupyter kernel (VS Code may not inherit pixi activation)
proj_data = os.path.join(os.getcwd(), ".pixi", "envs", "dev", "share", "proj")
if os.path.isdir(proj_data):
    os.environ["PROJ_DATA"] = proj_data

import geopandas as gpd
import matplotlib.pyplot as plt
import xarray as xr

## 1. Load fabric and temporal output

In [ ]:
fabric = gpd.read_file("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
ds_2020 = xr.open_dataset("../output/climate/gridmet_2020_temporal.nc")

print(f"Fabric HRU count: {len(fabric)}")
print(f"Temporal nhm_id count: {len(ds_2020.nhm_id)}")
print(f"Missing: {len(fabric) - len(ds_2020.nhm_id)}")

## 2. Identify missing features

In [ ]:
fabric_ids = set(fabric["nhm_id"].values)
temporal_ids = set(ds_2020.nhm_id.values)

missing_ids = sorted(fabric_ids - temporal_ids)
present_ids = sorted(fabric_ids & temporal_ids)

print(f"Missing IDs: {len(missing_ids)}")
print(f"Present IDs: {len(present_ids)}")
print(f"\nFirst 20 missing: {missing_ids[:20]}")
print(f"Last 20 missing: {missing_ids[-20:]}")

## 3. Map missing vs. present features

In [ ]:
fabric["status"] = fabric["nhm_id"].apply(
    lambda x: "missing" if x in set(missing_ids) else "present"
)

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
fabric.plot(
    column="status",
    categorical=True,
    legend=True,
    ax=ax,
    cmap="RdYlGn",
    edgecolor="gray",
    linewidth=0.3,
)
ax.set_title(f"Temporal Data Coverage: {len(present_ids)} present, {len(missing_ids)} missing")
plt.tight_layout()
plt.show()

## 4. Compare area distributions: missing vs. present

In [ ]:
# Compute areas in a projected CRS (fabric is EPSG:5070)
fabric["area_km2"] = fabric.geometry.area / 1e6

missing_areas = fabric.loc[fabric["status"] == "missing", "area_km2"]
present_areas = fabric.loc[fabric["status"] == "present", "area_km2"]

print("Missing features area stats (km²):")
print(missing_areas.describe())
print("\nPresent features area stats (km²):")
print(present_areas.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(missing_areas, bins=30, alpha=0.7, color="red", label="missing")
axes[0].hist(present_areas, bins=30, alpha=0.7, color="green", label="present")
axes[0].set_xlabel("Area (km²)")
axes[0].set_ylabel("Count")
axes[0].set_title("Area distribution")
axes[0].legend()

axes[1].hist(missing_areas, bins=30, alpha=0.7, color="red", label="missing")
axes[1].hist(present_areas, bins=30, alpha=0.7, color="green", label="present")
axes[1].set_xlabel("Area (km²)")
axes[1].set_ylabel("Count")
axes[1].set_title("Area distribution (log scale)")
axes[1].set_yscale("log")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Are missing features clustered spatially?

In [ ]:
# Color by area, with missing features highlighted
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: all features colored by area
fabric.plot(
    column="area_km2", legend=True, ax=axes[0], cmap="viridis", edgecolor="gray", linewidth=0.2
)
axes[0].set_title("All features by area (km²)")

# Right: missing features only
fabric[fabric["status"] == "present"].plot(
    ax=axes[1], color="lightgray", edgecolor="gray", linewidth=0.2
)
fabric[fabric["status"] == "missing"].plot(
    ax=axes[1], column="area_km2", legend=True, cmap="Reds", edgecolor="black", linewidth=0.5
)
axes[1].set_title(f"Missing features ({len(missing_ids)}) colored by area")

plt.tight_layout()
plt.show()

## 6. Check: are missing features smaller than gridMET resolution?

gridMET is ~4km (~16 km²). Small HRUs that don't fully contain a grid cell
centroid might get dropped by WeightGen.

In [ ]:
gridmet_cell_area_km2 = 16.0  # approx 4km x 4km

n_missing_small = (missing_areas < gridmet_cell_area_km2).sum()
n_present_small = (present_areas < gridmet_cell_area_km2).sum()

print(f"Features smaller than gridMET cell ({gridmet_cell_area_km2} km²):")
pct_missing = 100 * n_missing_small / len(missing_areas)
print(f"  Missing: {n_missing_small}/{len(missing_areas)} ({pct_missing:.0f}%)")
pct_present = 100 * n_present_small / len(present_areas)
print(f"  Present: {n_present_small}/{len(present_areas)} ({pct_present:.0f}%)")

## 7. Check static data for the same features

Do the same 134 features appear in the static SIR output?
If yes, this is specific to temporal/WeightGen processing.

In [ ]:
from pathlib import Path

import pandas as pd

sir_dir = Path("../output/sir")
sir_files = sorted(sir_dir.glob("*.csv"))
print(f"SIR files: {len(sir_files)}")

# Check the first SIR file for feature coverage
if sir_files:
    sample = pd.read_csv(sir_files[0], index_col=0)
    sir_ids = set(sample.index)
    print(f"\nSample SIR file: {sir_files[0].name}")
    print(f"  Features: {len(sir_ids)}")
    print(f"  Missing temporal IDs also missing in static: {len(set(missing_ids) - sir_ids)}")
    print(f"  Missing temporal IDs present in static: {len(set(missing_ids) & sir_ids)}")

## 8. Root Cause: Domain Bbox Clipping

**The 134 "missing" features are not a processing bug.** They are HRUs outside the
domain bbox configured in the pipeline YAML.

- Fabric file: 765 HRUs (full Delaware River Basin)
- Domain bbox `[-75.8, 39.6, -74.4, 42.5]`: clips to 631 HRUs
- The clipped set matches temporal output exactly
- The clipped set also matches static SIR output exactly

`stage1_resolve_fabric()` applies `gpd.clip(fabric, bbox_gdf)` before any
processing begins. All downstream stages (batching, static zonal stats,
temporal WeightGen) only see the 631 features within the bbox.

**No gap-filling needed for this case.** The pipeline is working correctly —
it processes exactly the features within the configured domain.

In [ ]:
from shapely.geometry import box

bbox_geom = box(-75.8, 39.6, -74.4, 42.5)
bbox_gdf = gpd.GeoDataFrame(index=[0], geometry=[bbox_geom], crs="EPSG:4326")
bbox_gdf = bbox_gdf.to_crs(fabric.crs)
clipped = gpd.clip(fabric, bbox_gdf)

print(f"Full fabric: {len(fabric)}")
print(f"After bbox clip: {len(clipped)}")
print(f"Clipped IDs == Temporal IDs: {set(clipped['nhm_id'].values) == temporal_ids}")
print("\nConclusion: All 'missing' features are outside the domain bbox.")
print("No batching or processing bug — the pipeline is working correctly.")